# S3 / MinIO Operations — Interactive Notebook

Companion notebook for [`s3_connection_operations.robot`](./s3_connection_operations.robot).
Each section below mirrors **one Robot test case** but uses raw boto3 calls so you can
tweak parameters interactively.

🔍 **How does this notebook relate to the Robot tests?** See [robot_vs_notebook.md](./robot_vs_notebook.md).

## 1:1 with the Robot suite

The 20 sections below match the 20 test cases in `s3_connection_operations.robot` exactly.
Section #1 (Create Account) has no boto3 equivalent — it's a SnapLogic platform call,
not an S3 operation — but the placeholder is kept for numbering parity.

## Prerequisites

- Run via `make jupyter-start` (it lives inside the tools container — boto3 ready).
- MinIO must be running: `make minio-start`.
- The bucket the cells reference will be created on the fly.

## 1. ACCOUNT — Create Account *(not applicable in notebook)*

The Robot test creates a **SnapLogic account** (a credential record on the SnapLogic platform)
via the SnapLogic API — not via S3 / boto3.

**Why no boto3 equivalent:** SnapLogic accounts are an orchestration-platform concept
(used by pipelines to connect to S3), not an S3 concept. Creating one requires the
SnapLogic admin API, which is out of scope for a notebook focused on S3 operations.

→ Keep using the Robot test for account creation. The remaining sections below
(2–19) all have direct boto3 equivalents.

## 2. CONNECT — Validate MinIO Connection

Equivalent to the `Validate MinIO Connection` Robot keyword. Creates an S3 client
and lists buckets as a connectivity test. If this fails, your credentials or
endpoint are wrong.

In [ ]:
import os
import boto3
from botocore.exceptions import ClientError

S3_ENDPOINT   = os.environ.get("S3_ENDPOINT",   "http://minio:9000")
S3_ACCESS_KEY = os.environ.get("S3_ACCESS_KEY", "minioadmin")
S3_SECRET_KEY = os.environ.get("S3_SECRET_KEY", "minioadmin")
S3_REGION     = os.environ.get("S3_REGION",     "us-east-1")

# Empty endpoint = real AWS S3. Anything else = MinIO or other S3-compatible.
s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT or None,
    aws_access_key_id=S3_ACCESS_KEY,
    aws_secret_access_key=S3_SECRET_KEY,
    region_name=S3_REGION,
)

response = s3.list_buckets()
buckets = [b["Name"] for b in response.get("Buckets", [])]
print(f"✅ Connected to {S3_ENDPOINT or '(AWS default)'}")
print(f"📦 Found {len(buckets)} bucket(s):")
for name in buckets:
    print(f"  - {name}")

## 3. BUCKET — Create Bucket (idempotent)

Equivalent to the `Create Bucket` Robot keyword. Catches `BucketAlreadyOwnedByYou`
and `BucketAlreadyExists` so re-running this cell is safe.

In [ ]:
BUCKET_NAME = "notebook-tutorial-bucket"
PREFIX      = "tutorial/"

try:
    s3.create_bucket(Bucket=BUCKET_NAME)
    print(f"✅ Bucket created: {BUCKET_NAME}")
except ClientError as e:
    code = e.response["Error"]["Code"]
    if code in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):
        print(f"ℹ️  Bucket already exists (idempotent): {BUCKET_NAME}")
    else:
        raise

# Idempotency demo — second call should be a no-op
try:
    s3.create_bucket(Bucket=BUCKET_NAME)
except ClientError as e:
    if e.response["Error"]["Code"] in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):
        print("✅ Second call also handled gracefully (idempotent)")
    else:
        raise

## 4. BUCKET — Check Bucket Exists

Equivalent to the `Check Bucket Exists` Robot keyword. Returns `True`/`False` —
does NOT raise an exception when the bucket is missing.

Uses `head_bucket` (cheap — metadata only, no listing).

In [ ]:
def bucket_exists(name: str) -> bool:
    try:
        s3.head_bucket(Bucket=name)
        return True
    except ClientError:
        return False

print(f"{BUCKET_NAME!r:35s} exists?  {bucket_exists(BUCKET_NAME)}")
print(f"{'nonexistent-bucket-xyz-999'!r:35s} exists?  {bucket_exists('nonexistent-bucket-xyz-999')}")

## 5. UPLOAD — Upload File To MinIO

Equivalent to the `Upload File To MinIO` Robot keyword. Uploads `sample_files/sample.txt`
(checked into the repo next to this notebook) to `s3://<bucket>/tutorial/sample.txt`.

In [ ]:
LOCAL_FILE = "./sample_files/sample.txt"
OBJECT_KEY = f"{PREFIX}sample.txt"

s3.upload_file(LOCAL_FILE, BUCKET_NAME, OBJECT_KEY)
print(f"📤 Uploaded {LOCAL_FILE} → s3://{BUCKET_NAME}/{OBJECT_KEY}")

## 6. EXISTENCE — Check Object Exists

Equivalent to the `Check Object Exists` Robot keyword. Uses `head_object` —
fast (metadata-only, no bytes downloaded).

In [ ]:
def object_exists(bucket: str, key: str) -> bool:
    try:
        s3.head_object(Bucket=bucket, Key=key)
        return True
    except ClientError:
        return False

print(f"{OBJECT_KEY!r:50s} exists?  {object_exists(BUCKET_NAME, OBJECT_KEY)}")
print(f"{'tutorial/missing.txt'!r:50s} exists?  {object_exists(BUCKET_NAME, 'tutorial/missing.txt')}")

## 7. METADATA — Get Object Metadata

Equivalent to the `Get Object Metadata` Robot keyword. Returns size, ETag,
content-type, last-modified, etc.

In [ ]:
metadata = s3.head_object(Bucket=BUCKET_NAME, Key=OBJECT_KEY)

print(f"Size (bytes):  {metadata['ContentLength']}")
print(f"Content-Type:  {metadata.get('ContentType', '(unknown)')}")
print(f"ETag:          {metadata['ETag']}")
print(f"LastModified:  {metadata['LastModified']}")

## 8. LIST — List Objects In Bucket

Equivalent to the `List Objects In Bucket` Robot keyword.
Tip: use the `Prefix` parameter to filter to one folder.

In [ ]:
response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=PREFIX)
contents = response.get("Contents", [])

print(f"📋 {len(contents)} object(s) under prefix {PREFIX!r}:")
for obj in contents:
    print(f"  - {obj['Key']:40s}  size={obj['Size']:>6}  modified={obj['LastModified']}")

## 9. SEARCH — Find Files In Bucket By Extension

Equivalent to the `Find Files In Bucket By Extension` Robot keyword. Lists objects,
filters by file extension, sorts descending so the latest is element [0].

The Robot keyword has an optional `path_filters` arg — we replicate that with a list.

In [ ]:
def find_files_by_extension(bucket: str, extension: str, path_filters=None):
    """Return (sorted_list, count) of objects matching the extension and any of the path_filters."""
    path_filters = path_filters or []      # empty list = match all paths
    response = s3.list_objects_v2(Bucket=bucket)
    matching = [
        obj["Key"]
        for obj in response.get("Contents", [])
        if obj["Key"].endswith(extension)
        and (not path_filters or any(pf in obj["Key"] for pf in path_filters))
    ]
    matching.sort(reverse=True)            # latest filename first
    return matching, len(matching)

files, count = find_files_by_extension(BUCKET_NAME, ".txt", path_filters=[PREFIX])
print(f"🔍 Found {count} '.txt' file(s) under {PREFIX!r}:")
for key in files:
    print(f"  - {key}")

## 10. SEARCH — Find Specific File In Bucket

Equivalent to the `Find Specific File In Bucket` Robot keyword. Returns
`(found_bool, candidates_list)` — useful for assertions plus debug context.

In [ ]:
def find_specific_file(bucket: str, expected_key: str, extension: str, path_filters=None):
    """Return (found, candidates) — True if expected_key is in the filtered list."""
    candidates, _ = find_files_by_extension(bucket, extension, path_filters)
    return expected_key in candidates, candidates

found, candidates = find_specific_file(
    BUCKET_NAME, OBJECT_KEY, ".txt", path_filters=[PREFIX]
)
print(f"🎯 Looking for: {OBJECT_KEY}")
print(f"   Found:        {found}")
print(f"   Candidates:   {candidates}")

## 11. DOWNLOAD — Download Single File From MinIO

Equivalent to the `Download Single File From MinIO` Robot keyword. Saves the
object to a local path; preserves the S3 key as the relative path under
`download_dir`.

In [ ]:
from pathlib import Path

DOWNLOAD_DIR = Path("/tmp/notebook_downloads")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

local_path = DOWNLOAD_DIR / OBJECT_KEY
local_path.parent.mkdir(parents=True, exist_ok=True)    # create tutorial/ subdir

s3.download_file(BUCKET_NAME, OBJECT_KEY, str(local_path))
print(f"📥 Downloaded → {local_path} ({local_path.stat().st_size} bytes)")

## 12. DOWNLOAD — Download And Get File Content

Equivalent to the `Download And Get File Content` Robot keyword — downloads
AND returns the file's content as a string. Convenient for assertions on
downloaded data.

In [ ]:
def download_and_get_content(bucket: str, download_dir: Path, key: str) -> str:
    local_path = download_dir / key
    local_path.parent.mkdir(parents=True, exist_ok=True)
    s3.download_file(bucket, key, str(local_path))
    return local_path.read_text()

content = download_and_get_content(BUCKET_NAME, DOWNLOAD_DIR, OBJECT_KEY)
print("📄 File content (first 200 chars):")
print(content[:200])

# Mirror Robot's marker assertion
assert "s3 tutorial sample" in content, "Marker phrase missing from downloaded content"
print("\n✅ Marker phrase 's3 tutorial sample' present")

## 13. UPLOAD — Upload Four More Files (for batch tests)

Equivalent to the `Upload Four More Files` Robot test. Loops over every
`.csv` and `.json` in `sample_files/`, uploads each to `tutorial/data/`.

In [ ]:
for file in sorted(Path("./sample_files").glob("*")):
    if file.suffix in (".csv", ".json"):
        key = f"{PREFIX}data/{file.name}"
        s3.upload_file(str(file), BUCKET_NAME, key)
        print(f"📤 {file.name:25s} → s3://{BUCKET_NAME}/{key}")

## 14. DOWNLOAD — Download Files By Pattern

Equivalent to the `Download Files By Pattern` Robot keyword. Downloads every
object whose key contains the given substring (not glob — plain substring match).

In [ ]:
def download_files_by_pattern(bucket: str, download_dir: Path, pattern: str) -> list:
    """Download every object whose key contains `pattern`. Returns the list of downloaded keys."""
    response = s3.list_objects_v2(Bucket=bucket)
    matching = [obj["Key"] for obj in response.get("Contents", []) if pattern in obj["Key"]]
    for key in matching:
        local_path = download_dir / key
        local_path.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(bucket, key, str(local_path))
    return matching

downloaded = download_files_by_pattern(BUCKET_NAME, DOWNLOAD_DIR, f"{PREFIX}data/")
print(f"📥 Downloaded {len(downloaded)} file(s) matching {PREFIX}data/:")
for key in downloaded:
    print(f"  - {key}")

## 15. DOWNLOAD — Download All Files From Bucket

Equivalent to the `Download All Files From Bucket` Robot keyword. Downloads
**every** object in the bucket — use with care; could be a lot of data on AWS.

In [ ]:
def download_all_from_bucket(bucket: str, download_dir: Path) -> list:
    response = s3.list_objects_v2(Bucket=bucket)
    keys = [obj["Key"] for obj in response.get("Contents", [])]
    for key in keys:
        local_path = download_dir / key
        local_path.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(bucket, key, str(local_path))
    return keys

all_downloaded = download_all_from_bucket(BUCKET_NAME, DOWNLOAD_DIR)
print(f"📦 Downloaded {len(all_downloaded)} object(s) from {BUCKET_NAME}:")
for key in all_downloaded:
    print(f"  - {key}")

## 16. VALIDATE — Verify All Files Are Non Empty In Bucket

Equivalent to the `Verify All Files Are Non Empty In Bucket` Robot keyword.
Iterates the given keys and asserts each has `Size > 0`.

In [ ]:
def verify_all_non_empty(bucket: str, keys: list) -> None:
    for key in keys:
        meta = s3.head_object(Bucket=bucket, Key=key)
        size = meta["ContentLength"]
        icon = "✅" if size > 0 else "❌"
        print(f"{icon} {key:40s}  {size:>6} bytes")
        assert size > 0, f"File '{key}' is empty (0 bytes)"

all_keys = [obj["Key"] for obj in s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=PREFIX).get("Contents", [])]
verify_all_non_empty(BUCKET_NAME, all_keys)

## 17. VALIDATE — Validate Downloaded File Template

Equivalent to the `Validate Downloaded File Template` Robot keyword. Checks
a *local* file (already downloaded): exists, meets minimum size, has expected
extension, and content is readable.

In [ ]:
def validate_downloaded_file(local_path: Path, min_size_bytes: int = 0, expected_extension: str = "") -> bool:
    assert local_path.exists(), f"File does not exist: {local_path}"
    size = local_path.stat().st_size
    assert size >= min_size_bytes, f"Size {size} < min {min_size_bytes}"
    if expected_extension:
        assert local_path.name.endswith(expected_extension), \
            f"Expected extension {expected_extension}, got {local_path.suffix}"
    content = local_path.read_text()
    assert content, "File is empty or unreadable"
    print(f"✅ Validated {local_path} — {size} bytes, extension OK, content readable")
    return True

validate_downloaded_file(
    DOWNLOAD_DIR / OBJECT_KEY,
    min_size_bytes=10,
    expected_extension=".txt",
)

## 18. CLEAN — Clean Bucket By Prefix

Equivalent to the `Clean Bucket By Prefix` Robot keyword. Deletes every object
whose key starts with the given prefix.

In [ ]:
to_delete = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=PREFIX).get("Contents", [])

for obj in to_delete:
    s3.delete_object(Bucket=BUCKET_NAME, Key=obj["Key"])
    print(f"🗑️  deleted {obj['Key']}")

print(f"\nDeleted {len(to_delete)} object(s) under prefix {PREFIX!r}.")

## 19. CLEAN — Delete Bucket (idempotent)

Equivalent to the `Delete Bucket` Robot keyword. Only succeeds if the bucket
is empty — the cleanup cell above ensures that. Idempotent — calling twice
doesn't fail.

In [ ]:
def delete_bucket_idempotent(name: str) -> None:
    try:
        s3.delete_bucket(Bucket=name)
        print(f"✅ Bucket deleted: {name}")
    except ClientError as e:
        code = e.response["Error"]["Code"]
        if code in ("NoSuchBucket", "NotFound"):
            print(f"ℹ️  Bucket already gone (idempotent): {name}")
        elif code == "BucketNotEmpty":
            print(f"⚠️  Bucket not empty — re-run cell 18 first.")
        else:
            raise

# First call — should succeed
delete_bucket_idempotent(BUCKET_NAME)

# Second call — bucket already gone, should still pass
delete_bucket_idempotent(BUCKET_NAME)

## 20. END TO END — Full S3 Workflow

Equivalent to the `END TO END — Full S3 Workflow` Robot test, which runs every
operation back-to-back as one chunk.

**The notebook is already an end-to-end run** when you execute every cell
top-to-bottom. To replay:

1. **Kernel → Restart Kernel & Clear All Outputs**
2. **Run → Run All Cells**

All 19 operations run in sequence — the bucket is created, files uploaded,
metadata fetched, downloads happen, prefix cleaned, bucket deleted. Same
behavior the Robot test produces in CI.

## What's next

- **Tweak `BUCKET_NAME` / `PREFIX`** at the top to point at your own bucket.
- **Point at AWS S3** by setting `S3_ENDPOINT=""` (empty) and exporting AWS keys.
- **Read the full Robot test suite** for keyword equivalents:
  [`s3_connection_operations.robot`](./s3_connection_operations.robot)
- **Read the keyword library** that wraps these calls:
  [`minio.resource`](../../../../../resources/minio/minio.resource)
- **Architecture relationship** between this notebook and the Robot tests:
  [`robot_vs_notebook.md`](./robot_vs_notebook.md)